# 5.4 硬件选型与基准测试方法论

产业级端侧AI部署的第一步是硬件选型，而选型的依据是系统化的基准测试。本章建立一套从硬件评估到选型决策的完整方法论，帮助工程师在众多端侧硬件中做出数据驱动的选择。

## 为什么需要系统化的硬件选型？

| 常见错误 | 后果 | 系统化选型的价值 |
|---------|------|-----------------|
| 凭品牌/参数选型 | 实际性能与标称值差距大 | 基于实测数据决策 |
| 只看峰值TOPS | 忽略内存带宽瓶颈 | Roofline分析定位真实瓶颈 |
| 单一指标对比 | 忽略功耗/成本/生态 | 多维度综合评估 |
| 一次性测试 | 忽略热节流和长期稳定性 | 长时间基准测试 |
| 只测小模型 | 大模型性能完全不同 | 目标模型实测 |

## 5.4.1 端侧硬件评估维度

### 五维评估框架

| 维度 | 关键指标 | 权重(推理) | 权重(训练) | 测量方法 |
|------|---------|-----------|-----------|----------|
| **性能** | TTFT, ITL, 吞吐, 峰值TOPS | 35% | 25% | 实际模型推理 |
| **内存** | 可用内存, 带宽, SRAM容量 | 25% | 30% | 硬件规格+实测 |
| **功耗** | TDP, 能效比, 电池影响 | 20% | 25% | 功耗仪测量 |
| **生态** | SDK成熟度, 算子覆盖, 社区 | 10% | 10% | 算子兼容性测试 |
| **成本** | 硬件成本, 开发成本, 维护成本 | 10% | 10% | BOM+人力估算 |

In [ ]:
from dataclasses import dataclass, field
from typing import List, Dict, Optional
import json

@dataclass
class HardwareCandidate:
    name: str
    vendor: str
    category: str
    npu_tops: float
    npu_tdp_watts: float
    dram_bandwidth_gbs: float
    available_memory_mb: int
    sram_kb: int
    supported_precisions: List[str]
    sdk_maturity: int
    hardware_cost_usd: float
    dev_effort_days: int

    @property
    def tops_per_watt(self):
        return self.npu_tops / self.npu_tdp_watts if self.npu_tdp_watts > 0 else 0

    @property
    def memory_bandwidth_per_watt(self):
        return self.dram_bandwidth_gbs / self.npu_tdp_watts if self.npu_tdp_watts > 0 else 0

CANDIDATES = [
    HardwareCandidate('Snapdragon 8 Gen3', 'Qualcomm', 'flagship_phone',
                      45, 6, 60, 12000, 4096, ['int4', 'int8', 'fp16'], 5, 800, 20),
    HardwareCandidate('Snapdragon 7 Gen3', 'Qualcomm', 'mid_phone',
                      25, 4, 40, 8000, 2048, ['int8', 'fp16'], 4, 400, 15),
    HardwareCandidate('Apple A17 Pro', 'Apple', 'flagship_phone',
                      35, 5, 50, 8000, 4096, ['fp16', 'int8'], 5, 900, 10),
    HardwareCandidate('Apple M4', 'Apple', 'tablet',
                      38, 8, 100, 16000, 4096, ['fp16', 'int8'], 5, 1200, 10),
    HardwareCandidate('Dimensity 9300', 'MediaTek', 'flagship_phone',
                      40, 5.5, 55, 12000, 3072, ['int8', 'fp16'], 3, 600, 25),
    HardwareCandidate('Ascend 310P', 'Huawei', 'edge_server',
                      22, 10, 51.2, 32768, 8192, ['int8', 'fp16'], 3, 300, 30),
    HardwareCandidate('Jetson Orin Nano', 'NVIDIA', 'edge_device',
                      40, 15, 68, 8192, 4096, ['int8', 'fp16', 'int4'], 5, 250, 15),
    HardwareCandidate('RK3588', 'Rockchip', 'sbc',
                      6, 5, 17, 8192, 1024, ['int8', 'fp16'], 2, 80, 35),
]

print("=== 端侧AI硬件候选列表 ===")
print(f"{'名称':<22} {'类别':<15} {'NPU TOPS':<10} {'TDP(W)':<8} {'TOPS/W':<8} {'内存(MB)':<10} {'BW(GB/s)':<10}")
print("-" * 85)
for c in CANDIDATES:
    print(f"{c.name:<22} {c.category:<15} {c.npu_tops:<10.0f} {c.npu_tdp_watts:<8.1f} "
          f"{c.tops_per_watt:<8.1f} {c.available_memory_mb:<10} {c.dram_bandwidth_gbs:<10.1f}")

## 5.4.2 基准测试方法论

### 四层基准测试体系

```
第一层: Micro-benchmark (微基准)
  → 测量单个算子(GEMM/Attention)的峰值性能
  → 确定Roofline模型的实际峰值
  → 耗时: 5-10分钟

第二层: Kernel-benchmark (核函数基准)
  → 测量完整推理步骤(prefill/decode)的性能
  → 验证算子融合和内存优化的效果
  → 耗时: 30-60分钟

第三层: Model-benchmark (模型基准)
  → 测量完整模型的端到端性能
  → 包含TTFT、吞吐、内存峰值、功耗
  → 耗时: 2-4小时

第四层: Stress-benchmark (压力基准)
  → 长时间运行(1-24小时)测试稳定性
  → 测量热节流、内存泄漏、性能退化
  → 耗时: 1-24小时
```

### 关键基准指标定义

| 指标 | 定义 | 测量方法 | 产业标准 |
|------|------|---------|----------|
| **TTFT** | 首 token 延迟 | 从输入到首个 token 输出 | <500ms(端侧) |
| **ITL** | token 间延迟 | 相邻两个 token 的时间间隔 | <50ms(端侧) |
| **吞吐** | tokens/second | 总生成 token 数/总时间 | >15 tok/s(可用) |
| **内存峰值** | 推理时最大内存占用 | 系统内存监控 | <设备内存70% |
| **功耗** | 平均推理功耗 | 功耗仪/系统API | <5W(手机) |
| **热节流比** | 节流后/节流前性能 | 长时间推理前后对比 | >70% |
| **P99延迟** | 99%请求的延迟上限 | 多次推理取P99 | <2x平均延迟 |

In [ ]:
class BenchmarkSuite:
    """四层基准测试套件模拟器"""
    def __init__(self, hardware: HardwareCandidate):
        self.hw = hardware

    def micro_benchmark(self):
        """第一层: 微基准测试"""
        peak_gemm_tops = self.hw.npu_tops * 0.7
        peak_bandwidth = self.hw.dram_bandwidth_gbs
        ridge_point = peak_gemm_tops * 1e12 / (peak_bandwidth * 1e9) / 1e9
        return {
            'peak_gemm_tflops': peak_gemm_tops,
            'peak_bandwidth_gbs': peak_bandwidth,
            'ridge_point_ops_per_byte': ridge_point,
            'sram_capacity_mb': self.hw.sram_kb / 1024,
        }

    def model_benchmark(self, model_params_b, seq_len, quant_bits=4):
        """第三层: 模型基准测试"""
        bytes_per_weight = quant_bits / 8
        model_size_mb = model_params_b * 1e9 * bytes_per_weight / 1e6
        fits_in_memory = model_size_mb < self.hw.available_memory_mb * 0.7
        if not fits_in_memory:
            return {'feasible': False, 'reason': 'memory_exceeded'}
        decode_bytes = model_size_mb * 1e6
        decode_latency_ms = decode_bytes / (self.hw.dram_bandwidth_gbs * 1e9 / 8) * 1000
        decode_tps = 1000 / decode_latency_ms
        prefill_flops = 2 * model_params_b * 1e9 * seq_len
        prefill_time_s = prefill_flops / (self.hw.npu_tops * 1e12 * 0.3)
        return {
            'feasible': True,
            'model_size_mb': model_size_mb,
            'ttft_ms': prefill_time_s * 1000,
            'decode_tps': decode_tps,
            'decode_latency_ms': decode_latency_ms,
        }

    def stress_benchmark(self, model_params_b, duration_minutes=30):
        """第四层: 压力基准测试"""
        tdp = self.hw.npu_tdp_watts
        thermal_throttle_ratio = max(0.5, 1.0 - (tdp - 5) * 0.03)
        sustained_tps_ratio = thermal_throttle_ratio
        return {
            'duration_min': duration_minutes,
            'initial_tps_ratio': 1.0,
            'sustained_tps_ratio': sustained_tps_ratio,
            'thermal_throttle_pct': (1 - thermal_throttle_ratio) * 100,
            'memory_leak_mb_per_hour': 0.5,
        }

print("=== 四层基准测试结果 ===")
for hw in CANDIDATES[:4]:
    suite = BenchmarkSuite(hw)
    micro = suite.micro_benchmark()
    model_3b = suite.model_benchmark(3, seq_len=512, quant_bits=4)
    model_7b = suite.model_benchmark(7, seq_len=512, quant_bits=4)
    stress = suite.stress_benchmark(3)
    print(f"\n--- {hw.name} ---")
    print(f"  微基准: 峰值GEMM={micro['peak_gemm_tflops']:.0f}TOPS, 带宽={micro['peak_bandwidth_gbs']:.0f}GB/s, Ridge={micro['ridge_point_ops_per_byte']:.1f} ops/byte")
    if model_3b['feasible']:
        print(f"  3B模型: TTFT={model_3b['ttft_ms']:.0f}ms, Decode={model_3b['decode_tps']:.1f}tok/s")
    else:
        print(f"  3B模型: 不可行({model_3b['reason']})")
    if model_7b['feasible']:
        print(f"  7B模型: TTFT={model_7b['ttft_ms']:.0f}ms, Decode={model_7b['decode_tps']:.1f}tok/s")
    else:
        print(f"  7B模型: 不可行({model_7b['reason']})")
    print(f"  压力测试: 持续性能比={stress['sustained_tps_ratio']:.0%}, 热节流={stress['thermal_throttle_pct']:.0%}")

## 5.4.3 硬件选型决策框架

### 选型决策流程

```
① 确定需求
   → 目标模型大小、延迟要求、功耗预算、成本上限
   ↓
② 初筛候选
   → 根据内存和算力要求过滤不可行方案
   ↓
③ 基准测试
   → 四层基准测试，获取实测数据
   ↓
④ 多维评分
   → 五维评估框架，加权打分
   ↓
⑤ 风险评估
   → 供应链风险、SDK成熟度、长期支持
   ↓
⑥ 最终决策
   → 综合评分+风险评估，选择最优方案
```

### 不同场景的选型策略

| 场景 | 性能要求 | 功耗要求 | 成本要求 | 推荐硬件类型 |
|------|---------|---------|---------|-------------|
| **旗舰手机** | 高(7B@15tok/s) | <5W | 高端 | 骁龙8 Gen3 / A17 Pro |
| **中端手机** | 中(3B@15tok/s) | <4W | 中端 | 骁龙7 Gen3 / 天玑8300 |
| **平板/PC** | 高(7B@30tok/s) | <15W | 中高端 | M4 / 骁龙X Elite |
| **边缘服务器** | 很高(13B@20tok/s) | <30W | 中端 | Jetson Orin / Ascend 310P |
| **IoT/SBC** | 低(1.5B@5tok/s) | <5W | 低成本 | RK3588 / ESP32(极小模型) |
| **车载** | 高(7B@20tok/s) | <15W | 高端 | 骁龙8 Gen3 / Orin NX |

In [ ]:
class HardwareSelectionAdvisor:
    """硬件选型决策顾问"""
    def __init__(self, candidates: List[HardwareCandidate]):
        self.candidates = candidates

    def evaluate(self, model_params_b, target_tps, max_power_w, max_cost_usd,
                 weights=None):
        if weights is None:
            weights = {'performance': 0.35, 'memory': 0.25, 'power': 0.20,
                       'ecosystem': 0.10, 'cost': 0.10}
        results = []
        for hw in self.candidates:
            suite = BenchmarkSuite(hw)
            model_result = suite.model_benchmark(model_params_b, seq_len=512, quant_bits=4)
            stress = suite.stress_benchmark(model_params_b)
            if not model_result['feasible']:
                results.append({'hardware': hw.name, 'feasible': False, 'score': 0})
                continue
            perf_score = min(model_result['decode_tps'] / target_tps, 2.0) / 2.0
            mem_score = min(hw.available_memory_mb / (model_result['model_size_mb'] / 0.7), 1.0)
            power_score = 1.0 - min(hw.npu_tdp_watts / max_power_w, 1.0)
            eco_score = hw.sdk_maturity / 5.0
            cost_score = 1.0 - min(hw.hardware_cost_usd / max_cost_usd, 1.0)
            total = (weights['performance'] * perf_score +
                     weights['memory'] * mem_score +
                     weights['power'] * power_score +
                     weights['ecosystem'] * eco_score +
                     weights['cost'] * cost_score)
            sustained_tps = model_result['decode_tps'] * stress['sustained_tps_ratio']
            results.append({
                'hardware': hw.name,
                'feasible': True,
                'score': total,
                'decode_tps': model_result['decode_tps'],
                'sustained_tps': sustained_tps,
                'ttft_ms': model_result['ttft_ms'],
                'power_w': hw.npu_tdp_watts,
                'cost_usd': hw.hardware_cost_usd,
                'perf_score': perf_score,
                'mem_score': mem_score,
                'power_score': power_score,
                'eco_score': eco_score,
                'cost_score': cost_score,
            })
        results.sort(key=lambda x: x['score'], reverse=True)
        return results

advisor = HardwareSelectionAdvisor(CANDIDATES)

print("=== 硬件选型决策: 3B模型, 目标15 tok/s, <8W, <$1000 ===")
results = advisor.evaluate(model_params_b=3, target_tps=15, max_power_w=8, max_cost_usd=1000)
print(f"\n{'排名':<4} {'硬件':<22} {'综合分':<8} {'Decode':<10} {'持续':<10} {'TTFT':<10} {'功耗':<8} {'成本'}")
print("-" * 80)
for i, r in enumerate(results):
    if not r['feasible']:
        print(f"{i+1:<4} {r['hardware']:<22} 不可行")
    else:
        print(f"{i+1:<4} {r['hardware']:<22} {r['score']:<8.2f} {r['decode_tps']:<10.1f} "
              f"{r['sustained_tps']:<10.1f} {r['ttft_ms']:<10.0f} {r['power_w']:<8.1f} ${r['cost_usd']}")

print(f"\n=== 硬件选型决策: 7B模型, 目标10 tok/s, <10W, <$1500 ===")
results_7b = advisor.evaluate(model_params_b=7, target_tps=10, max_power_w=10, max_cost_usd=1500)
print(f"\n{'排名':<4} {'硬件':<22} {'综合分':<8} {'Decode':<10} {'持续':<10} {'TTFT':<10} {'功耗':<8} {'成本'}")
print("-" * 80)
for i, r in enumerate(results_7b):
    if not r['feasible']:
        print(f"{i+1:<4} {r['hardware']:<22} 不可行")
    else:
        print(f"{i+1:<4} {r['hardware']:<22} {r['score']:<8.2f} {r['decode_tps']:<10.1f} "
              f"{r['sustained_tps']:<10.1f} {r['ttft_ms']:<10.0f} {r['power_w']:<8.1f} ${r['cost_usd']}")

## 5.4.4 基准测试的工程实践

### 基准测试的常见陷阱

| 陷阱 | 表现 | 正确做法 |
|------|------|----------|
| **冷启动偏差** | 首次推理慢(mmap未加载) | 丢弃前3-5次warmup结果 |
| **热节流未考虑** | 短测快、长测慢 | 至少30分钟压力测试 |
| **后台进程干扰** | 结果波动大 | 关闭后台应用，隔离测试 |
| **只测平均** | 忽略长尾延迟 | 报告P50/P90/P99 |
| **量化配置不一致** | 不同硬件用不同量化 | 统一量化配置对比 |
| **模型版本不一致** | 对比不公平 | 使用同一模型checkpoint |

### 基准测试报告模板

```markdown
## 基准测试报告
- **日期**: YYYY-MM-DD
- **硬件**: [设备型号, OS版本, SDK版本]
- **模型**: [模型名, 参数量, 量化配置]
- **测试配置**: [seq_len, batch_size, 重复次数]

### 结果
| 指标 | 值 | 备注 |
|------|------|------|
| TTFT (P50/P90/P99) | xx/xx/xx ms | |
| ITL (P50/P90/P99) | xx/xx/xx ms | |
| 吞吐 | xx tok/s | |
| 内存峰值 | xx MB | |
| 平均功耗 | xx W | |
| 热节流比 | xx% | 30min后 |
```

### 产业实践要点

1. **基准测试是持续过程**：每次SDK/模型更新后必须重新测试
2. **自动化基准测试**：CI/CD集成，每次提交自动跑benchmark
3. **建立性能数据库**：历史数据追踪，发现性能回归
4. **多设备采样**：同型号设备间也有差异，至少测3台
5. **报告P99而非平均**：用户体验由长尾决定
6. **功耗测试用外接功耗仪**：系统API报告的功耗不准确

In [ ]:
class BenchmarkReporter:
    """基准测试报告生成器"""
    def __init__(self, hardware_name, model_name, config):
        self.hardware_name = hardware_name
        self.model_name = model_name
        self.config = config
        self.results = []

    def add_run(self, ttft_ms, itl_ms, memory_mb, power_w):
        self.results.append({
            'ttft_ms': ttft_ms, 'itl_ms': itl_ms,
            'memory_mb': memory_mb, 'power_w': power_w
        })

    def generate_report(self):
        import numpy as np
        if not self.results:
            return "无测试数据"
        ttfts = [r['ttft_ms'] for r in self.results]
        itls = [r['itl_ms'] for r in self.results]
        memories = [r['memory_mb'] for r in self.results]
        powers = [r['power_w'] for r in self.results]
        return {
            'hardware': self.hardware_name,
            'model': self.model_name,
            'n_runs': len(self.results),
            'ttft': {'p50': np.percentile(ttfts, 50), 'p90': np.percentile(ttfts, 90),
                     'p99': np.percentile(ttfts, 99)},
            'itl': {'p50': np.percentile(itls, 50), 'p90': np.percentile(itls, 90),
                    'p99': np.percentile(itls, 99)},
            'throughput_tps': 1000 / np.percentile(itls, 50),
            'memory_peak_mb': max(memories),
            'power_avg_w': np.mean(powers),
        }

reporter = BenchmarkReporter('Snapdragon 8 Gen3', 'Qwen2.5-3B-Instruct-Q4',
                              {'seq_len': 512, 'n_runs': 100})
np.random.seed(42)
for _ in range(100):
    ttft = np.random.normal(250, 30)
    itl = np.random.normal(45, 8)
    memory = np.random.normal(2800, 50)
    power = np.random.normal(4.5, 0.3)
    reporter.add_run(ttft, itl, memory, power)

report = reporter.generate_report()
print(f"=== 基准测试报告 ===")
print(f"硬件: {report['hardware']}")
print(f"模型: {report['model']}")
print(f"测试次数: {report['n_runs']}")
print(f"\n--- 延迟指标 ---")
print(f"TTFT: P50={report['ttft']['p50']:.0f}ms, P90={report['ttft']['p90']:.0f}ms, P99={report['ttft']['p99']:.0f}ms")
print(f"ITL:  P50={report['itl']['p50']:.0f}ms, P90={report['itl']['p90']:.0f}ms, P99={report['itl']['p99']:.0f}ms")
print(f"吞吐: {report['throughput_tps']:.1f} tok/s (基于P50 ITL)")
print(f"\n--- 资源指标 ---")
print(f"内存峰值: {report['memory_peak_mb']:.0f} MB")
print(f"平均功耗: {report['power_avg_w']:.1f} W")

print(f"\n=== 硬件选型总结 ===")
print(f"1. 系统化选型 = 需求定义 + 基准测试 + 多维评分 + 风险评估")
print(f"2. 四层基准测试: 微基准→核函数→模型→压力，逐层深入")
print(f"3. 五维评估: 性能35% + 内存25% + 功耗20% + 生态10% + 成本10%")
print(f"4. 永远报告P99而非平均，用户体验由长尾决定")
print(f"5. 热节流是常态，30分钟+压力测试不可省略")
print(f"6. 基准测试自动化集成到CI/CD，防止性能回归")